# Módulo 50: Publicación de dashboards — Solución
*Juliho Castillo*

Enunciado para estudiantes: [m50-publicacion-de-dashboards-guia.ipynb](m50-publicacion-de-dashboards-guia.ipynb) · Lección: [m50-publicacion-de-dashboards-leccion.ipynb](m50-publicacion-de-dashboards-leccion.ipynb)

## Contexto

La app de ventas del Módulo 49 se prepara para publicarla en Streamlit Community Cloud y se le añade seguridad por filas: un rol `NAFTA` que solo ve Estados Unidos, Canadá y México, y una tabla que asigna países a cada correo. Los archivos están en [`../practica-ventas/`](../practica-ventas/) ([`requirements.txt`](../practica-ventas/requirements.txt), [`seguridad.py`](../practica-ventas/seguridad.py), [`comun.py`](../practica-ventas/comun.py)) y en la carpeta `.streamlit/` de la raíz ([`secrets.example.toml`](../../.streamlit/secrets.example.toml)). El enunciado original del curso se perdió; las instrucciones de la guía se reconstruyeron a partir de la solución.

In [1]:
import sys
import tomllib
from pathlib import Path

import streamlit.config
import streamlit.logger
from streamlit.testing.v1 import AppTest

# Sin mensajes del registro de Streamlit en las salidas (los errores se leen con at.exception)
streamlit.config.set_option("logger.level", "critical")
streamlit.logger.set_log_level("CRITICAL")

RAIZ = Path("../..").resolve()
PRACTICA = Path("../practica-ventas").resolve()
sys.path.insert(0, str(PRACTICA))
import estilo
import seguridad
import ventas as practica

con = practica.conectar()
APP = str(PRACTICA / "app.py")
NAFTA = ["US", "CANADA", "MEXICO"]

### Tarea 1. Requisitos para publicar

Según la documentación de Streamlit Community Cloud (consultada el 18 de octubre de 2026):

1. Hacen falta una cuenta de **GitHub**, con el código en un repositorio, y una cuenta de Community Cloud (se entra con GitHub). El formulario pide repositorio, rama y ruta del archivo principal, y opcionalmente el subdominio de la URL; en la configuración avanzada se elige la versión de Python (3.12 por defecto) y se pegan los secretos.
2. Reconoce `uv.lock`, `Pipfile`, `environment.yml`, `requirements.txt` y `pyproject.toml` (poetry), en ese orden de prioridad dentro de cada carpeta. Busca **primero en la carpeta del archivo principal y después en la raíz**. Con este repositorio, un `requirements.txt` en `practica-ventas/` gana a `uv.lock` y `pyproject.toml` de la raíz, porque está junto a `app.py`.
3. La app se ejecuta siempre desde la **raíz del repositorio**, y `.streamlit/config.toml` debe estar en la raíz.

### Tarea 2. Preparar el despliegue

In [2]:
import ast
import subprocess

print((PRACTICA / "requirements.txt").read_text(encoding="utf-8"))

LOCALES = {"comun", "estilo", "seguridad", "ventas"}
archivos = [PRACTICA / "app.py", *PRACTICA.glob("*.py"), *(PRACTICA / "paginas").glob("*.py")]
externas = set()
for archivo in archivos:
    for nodo in ast.walk(ast.parse(archivo.read_text(encoding="utf-8"))):
        if isinstance(nodo, ast.Import):
            externas |= {alias.name.split(".")[0] for alias in nodo.names}
        elif isinstance(nodo, ast.ImportFrom) and nodo.level == 0:
            externas.add(nodo.module.split(".")[0])
externas -= LOCALES | set(sys.stdlib_module_names) | {"__future__"}

lineas = (PRACTICA / "requirements.txt").read_text(encoding="utf-8").splitlines()
lista = {l.split("[")[0].split(">")[0].split("<")[0].split("=")[0].strip()
         for l in lineas if l.strip() and not l.startswith("#")}
print("Importa:", sorted(externas), "| faltan en requirements.txt:", sorted(externas - lista) or "ninguna")

# Dependencias de la app para Streamlit Community Cloud (Módulo 50).
# Community Cloud busca el archivo de dependencias primero junto al archivo
# principal (esta carpeta) y después en la raíz del repositorio, así que este
# archivo tiene prioridad sobre pyproject.toml y uv.lock de la raíz.
streamlit>=1.66
plotly>=6.0
duckdb>=1.2
pandas>=2.2,<3
pyarrow>=17
# Para iniciar sesión con st.login() (proveedor OIDC), sustituye la primera línea por:
# streamlit[auth]>=1.66

Importa: ['duckdb', 'pandas', 'plotly', 'streamlit'] | faltan en requirements.txt: ninguna


In [3]:
for archivo in sorted((PRACTICA / "datos-modelo").glob("*.parquet")):
    filtro = subprocess.run(["git", "check-attr", "filter", "--", archivo.name], cwd=archivo.parent,
                            capture_output=True, text=True).stdout.strip().rsplit(": ", 1)[-1]
    print(f"{archivo.name:18} {archivo.stat().st_size / 1e6:5.2f} MB | filtro: {filtro}")

# Rutas absolutas o barras invertidas en el código de la app
sospechosas = [(a.name, n) for a in archivos for n, linea in enumerate(a.read_text(encoding="utf-8").splitlines(), 1)
               if ":\\" in linea or "C:/" in linea]
print("Rutas absolutas:", sospechosas or "ninguna")

clientes.parquet    0.04 MB | filtro: unspecified
paises.parquet      0.00 MB | filtro: unspecified


productos.parquet   0.02 MB | filtro: unspecified
tiendas.parquet     0.01 MB | filtro: unspecified


ventas.parquet      8.51 MB | filtro: unspecified
Rutas absolutas: ninguna


`requirements.txt` tiene cinco librerías con versiones mínimas (Streamlit 1.66, la versión con la que se probó la app; `pandas<3`) y cubre todo lo que la app importa. Los Parquet no pasan por Git LFS (`unspecified`), el mayor ocupa 8.51 MB y el código no tiene rutas absolutas: `ventas.py` construye las rutas con `Path(__file__)`.

### Tarea 3. Secretos

In [4]:
ignorados = (RAIZ / ".gitignore").read_text(encoding="utf-8").splitlines()
print("secrets.toml en .gitignore:", ".streamlit/secrets.toml" in ignorados)
print("secrets.toml en disco:", (RAIZ / ".streamlit" / "secrets.toml").exists())
ejemplo = tomllib.loads((RAIZ / ".streamlit" / "secrets.example.toml").read_text(encoding="utf-8"))
print("Secciones del ejemplo:", list(ejemplo))

# Sin archivo de secretos, la app arranca en modo demostración
at = AppTest.from_file(APP, default_timeout=90).run()
print("Excepciones sin secretos:", len(at.exception), "| selector:", at.selectbox(key="ver_como").label)

secrets.toml en .gitignore: True
secrets.toml en disco: False
Secciones del ejemplo: ['roles', 'miembros', 'usuarios', 'acceso']


Excepciones sin secretos: 0 | selector: Ver como (solo demo)


El archivo real está ignorado por git y no existe en el repositorio; el de ejemplo documenta `[roles]`, `[miembros]`, `[usuarios]` y `[acceso]`, con `[auth]` comentado. `seguridad.secretos()` envuelve `st.secrets` en un `try`, de modo que sin archivo la app arranca en modo demostración. En Community Cloud el mismo contenido TOML se pega en el campo **Secrets** de la configuración avanzada, al desplegar o después.

### Tarea 4. Rol de seguridad NAFTA

In [5]:
print("Regla del rol:", seguridad.paises_de_opcion("rol:NAFTA", ejemplo))
ventas_nafta = practica.kpis(con, paises=NAFTA)["ventas_total"]
print("Ventas del rol NAFTA:", estilo.entero(ventas_nafta), f"({ventas_nafta / practica.VENTAS_TOTALES:.2%})")
print(practica.ventas_por(con, "pais", paises=NAFTA)[["pais", "ventas_total"]].to_string(index=False))

Regla del rol: ['US', 'CANADA', 'MEXICO']
Ventas del rol NAFTA: 16 364 381 054 (39.02%)
  pais  ventas_total
CANADA    6683524091
    US    5025305951
MEXICO    4655551012


El rol se define en `[roles]` y `comun.filtros()` recorta el argumento `paises` de todas las consultas a la lista permitida, así que DuckDB nunca devuelve filas de otros países. El rol NAFTA debe ver 16 364 381 054, el 39.02 % de las ventas.

### Tarea 5. Seguridad por filas dinámica

In [6]:
for correo in ["jorge@empresa.com", "marta@empresa.com", "direccion@empresa.com", "nadie@empresa.com"]:
    print(f"{correo:22} -> {seguridad.paises_de_correo(correo, ejemplo)}")

jorge@empresa.com      -> ['MEXICO', 'US']
marta@empresa.com      -> ['FRANCE', 'PORTUGAL', 'SPAIN']
direccion@empresa.com  -> None
nadie@empresa.com      -> []


La tabla `[usuarios]` asigna países a cada correo; `paises_de_correo` les suma los de sus roles (`[miembros]`), da `None` (todo) a los administradores y `[]` (nada) a un correo desconocido. En producción el correo sale de `st.user.email` después de `st.login()` con un proveedor OIDC configurado en `[auth]`; un cuadro de texto no sirve porque cualquiera podría escribir el correo de otra persona. Esta vía no se pudo probar porque requiere credenciales reales de un proveedor de identidad.

### Tarea 6. Probar "ver como"

In [7]:
at = AppTest.from_file(APP, default_timeout=90)
at.secrets["roles"] = ejemplo["roles"]
at.secrets["usuarios"] = ejemplo["usuarios"]
at.run()
print("Opciones:", at.selectbox(key="ver_como").options)

ventas_jorge = practica.kpis(con, paises=["US", "MEXICO"])["ventas_total"]
for opcion, esperado in [("rol:NAFTA", ventas_nafta), ("usuario:jorge@empresa.com", ventas_jorge)]:
    at.selectbox(key="ver_como").select(opcion).run()
    at.switch_page("paginas/resumen.py").run()
    paises = sorted(at.dataframe[0].value["pais"])
    print(f"{opcion:26} {at.metric[0].value:>12} (esperado {estilo.millones(esperado)}) | países: {paises}")
    assert at.metric[0].value == estilo.millones(esperado)

Opciones: ['Todos los países (sin restricción)', 'Rol NAFTA', 'Rol SUDAMERICA', 'Rol EUROPA', 'Usuario marta@empresa.com', 'Usuario jorge@empresa.com']


rol:NAFTA                   16 364.38 M (esperado 16 364.38 M) | países: ['CANADA', 'MEXICO', 'US']


usuario:jorge@empresa.com    9 680.86 M (esperado 9 680.86 M) | países: ['MEXICO', 'US']


In [8]:
# El CSV que descarga el resumen sale de la misma consulta filtrada
at.selectbox(key="ver_como").select("rol:NAFTA").run()
at.switch_page("paginas/resumen.py").run()
filtro = {"paises": tuple(at.session_state["paises_permitidos"]), "categorias": None,
          "desde": at.session_state["desde"], "hasta": at.session_state["hasta"]}
csv = practica.tabla_por_pais(con, **filtro).to_csv(index=False)
print(csv)
print("Botón de descarga:", [d.proto.label for d in at.get("download_button")])

pais,iso3,ventas_mediana,ventas_total,cantidad_total,pct_cantidad,ventas_promedio
CANADA,CAN,22954.0,6683524091,14809154,0.3334042129796114,44736.36923519726
US,USA,18060.0,5025305951,14889285,0.3352082331815939,33406.49709165121
MEXICO,MEX,14630.0,4655551012,14719578,0.33138755383879476,31095.050841570934

Botón de descarga: ['Descargar tabla (CSV)']


Con "Ver como" en el rol NAFTA, la tarjeta muestra 16 364.38 M y la tabla solo `CANADA`, `MEXICO` y `US`; con `jorge@empresa.com`, 9 680.86 M y solo `MEXICO` y `US`. El CSV que descarga el resumen se genera con la misma consulta filtrada, así que tampoco contiene otros países, y su porcentaje de unidades suma 100 % entre los tres.

### Tarea 7 (opcional). Publicar en Community Cloud

No se publicó: requiere cuentas propias. Los pasos serían: en share.streamlit.io, **Create app** → repositorio, rama `main` y archivo principal `s08-dashboards/practica-ventas/app.py`; en **Advanced settings**, Python 3.11 o 3.12 y el contenido de los secretos; **Deploy**. Si la app fuera privada (repositorio privado), el gerente se añadiría como visor desde **Share**, con su correo; para que además vea solo sus países, su correo debe estar en `[usuarios]` o en un rol de `[miembros]` y la app debe tener `[auth]` configurado para que `st.login()` confirme su identidad.

**Revisión posterior (2026-10):** El ejercicio se resolvió originalmente en Power BI (requisitos de licencia de Power BI Service, rol de seguridad de nivel de fila `NAFTA` en DAX y exportación a PDF con "Ver como"). Se rehízo en Python con Streamlit (Community Cloud, `st.secrets`, `st.login`/`st.user`) y DuckDB para que la sección no dependa de software de pago ni de Windows; la versión original sigue en el historial de git.